In [0]:
from pyspark.sql.functions import *

from delta.tables import DeltaTable

In [0]:
base_adls2_path = "abfss://customer360unitycatalog@kmstorage9490.dfs.core.windows.net"

In [0]:
source_df = spark.sql('select * from customer360_cata.bronze.payment_types')

source_df = source_df.withColumn('create_at',current_timestamp()) \
            .withColumn('updated_at',current_timestamp())

In [0]:


target_tbl = "customer360_cata.silver.payment_types"

# =====================================================
# TABLE EXISTS
# =====================================================

if spark.catalog.tableExists(target_tbl):

    print("Table exists")

    delta_table = DeltaTable.forName(spark, target_tbl)
    
    # =================================================
    # MERGE
    # =================================================

    (
        delta_table.alias("t")
        .merge(
            source_df.alias("s"),
             "t.payment_id = s.payment_id"
        )
       .whenMatchedUpdate(
            condition="""
                t.payment_method <> s.payment_method
            """,
            set={
                "payment_method": "s.payment_method",
                "updated_at":current_timestamp()
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("SCD Type 1 Merge Completed")

else:

    print("Table does not exist")

    source_df.write \
        .format("delta") \
        .mode("overwrite") \
            .option("overwriteSchema", "true") \
        .option("path", f"{base_adls2_path}/silver/payment_types") \
        .saveAsTable(target_tbl)

    print("Initial Load Completed")

In [0]:
%sql
--select * from customer360_cata.silver.payment_types  

In [0]:
%sql
 --drop table if exists customer360_cata.silver.payment_types